# CDS 303 Final Project: Data Reference Notebook

**Team 8:** Dae Yeong Mun, Oscar Molina Romualdo



Datasets and their sources are listed on the project website: https://daeyeong-mun.github.io/CDS-303-Final-project/Supplementary%20Dataset.html


In [22]:
import pandas as pd

### English Translation Setup

The source files from the Seoul Open Data Portal use Korean column names, and a few columns use Korean category values or Korean place names. Everything below is translated to English immediately after loading, so every table and output in this notebook — from here on — is in English.

**Columns**

| Korean | English |
|---|---|
| 자전거번호 | bike_id |
| 대여일시 | rental_datetime |
| 대여 대여소번호 | rental_station_number |
| 대여 대여소명 | rental_station_name_kr |
| 대여거치대 | rental_dock_number |
| 반납일시 | return_datetime |
| 반납대여소번호 | return_station_number |
| 반납대여소명 | return_station_name_kr |
| 반납거치대 | return_dock_number |
| 이용시간(분) | duration_min |
| 이용거리(M) | distance_m |
| 생년 | birth_year |
| 성별 | gender |
| 이용자종류 | user_type |
| 대여대여소ID | rental_station_id |
| 반납대여소ID | return_station_id |
| 대여일자 (daily summary file) | rental_date |
| 대여건수 (daily summary file) | rental_count |

**Category values**: 내국인 → Domestic, 외국인 → Foreigner, 비회원 → Non-member; M/m → Male, F/f → Female.

**Station names** are Korean place names (subway exits, landmarks), so rather than a meaning-based "translation" we romanize them — the same Korean-to-Latin-alphabet spelling system used on Korean street signs — computed directly from the Hangul characters below (no external service, works for all 2,761 stations).

In [23]:
## English column-name and category-value mappings
COLUMN_TRANSLATIONS = {
    "자전거번호": "bike_id",
    "대여일시": "rental_datetime",
    "대여 대여소번호": "rental_station_number",
    "대여 대여소명": "rental_station_name_kr",
    "대여거치대": "rental_dock_number",
    "반납일시": "return_datetime",
    "반납대여소번호": "return_station_number",
    "반납대여소명": "return_station_name_kr",
    "반납거치대": "return_dock_number",
    "이용시간(분)": "duration_min",
    "이용거리(M)": "distance_m",
    "생년": "birth_year",
    "성별": "gender",
    "이용자종류": "user_type",
    "대여대여소ID": "rental_station_id",
    "반납대여소ID": "return_station_id",
    "대여일자": "rental_date",
    "대여건수": "rental_count",
    "지점": "station_id",
    "지점명": "station_name_kr",
    "일시": "date",
    "평균기온(°C)": "avg_temp_c",
    "최저기온(°C)": "min_temp_c",
    "최고기온(°C)": "max_temp_c",
    "일강수량(mm)": "precip_mm",
    "평균 풍속(m/s)": "avg_wind_mps",
    "평균 상대습도(%)": "avg_humidity_pct",
    "합계 일조시간(hr)": "sunshine_hr",
}

GENDER_MAP = {"M": "Male", "m": "Male", "F": "Female", "f": "Female"}
USER_TYPE_MAP = {"내국인": "Domestic", "외국인": "Foreigner", "비회원": "Non-member"}

def to_english(df):
    """Rename columns to English and translate gender/user_type values, if present."""
    df = df.rename(columns=COLUMN_TRANSLATIONS)
    if "gender" in df.columns:
        df["gender"] = df["gender"].map(GENDER_MAP).fillna(df["gender"])
    if "user_type" in df.columns:
        df["user_type"] = df["user_type"].map(USER_TYPE_MAP).fillna(df["user_type"])
    return df

In [24]:
## Romanize Korean station names (Revised Romanization, pure Python)
CHO = ["g","kk","n","d","tt","r","m","b","pp","s","ss","","j","jj","ch","k","t","p","h"]
JUNG = ["a","ae","ya","yae","eo","e","yeo","ye","o","wa","wae","oe","yo","u","wo","we","wi","yu","eu","ui","i"]
JONG = ["","k","k","k","n","n","n","t","l","k","m","l","l","l","p","l","m","p","p","t","t","ng","t","t","k","t","p","t"]

def romanize_syllable(ch):
    code = ord(ch)
    if 0xAC00 <= code <= 0xD7A3:
        idx = code - 0xAC00
        return CHO[idx // (21*28)] + JUNG[(idx % (21*28)) // 28] + JONG[idx % 28]
    return ch

def romanize(text):
    if pd.isna(text):
        return text
    return "".join(romanize_syllable(c) for c in str(text)).title()

## Daily Rental Summary

In [25]:
df = to_english(pd.read_csv("서울특별시 공공자전거 일별 대여건수_26.1-6.csv", encoding="cp949"))

In [26]:
## First Few Rows
df.head()

,rental_date,rental_count
0,2026-01-01,27245
1,2026-01-02,47606
2,2026-01-03,41385
3,2026-01-04,42743
4,2026-01-05,68116


In [27]:
## Dataset Dimensions
df.shape

(181, 2)

In [28]:
## Column Names
df.columns

Index(['rental_date', 'rental_count'], dtype='str')

In [29]:
## Data Types
df.dtypes

rental_date       str
rental_count    int64
dtype: object

In [30]:
## Basic Summary Statistics
df.describe()

,rental_count
count,181.000000
mean,99331.441989
std,41517.757494
min,9975.000000
25%,62125.000000
50%,102124.000000
75%,138004.000000
max,158650.000000


In [31]:
## Missing Values
df.isnull().sum()

rental_date     0
rental_count    0
dtype: int64

## Trip-Level Rental History (Jan–Jun 2026)

The main dataset: one combined CSV of about 18.5 million individual Ttareungi trips (~3.1 GB, CP949-encoded) from Seoul Open Data Plaza (OA-15182). Each row records the rental and return station, timestamps, trip duration and distance, and rider details.

The file is too large to load comfortably in one go, so later analysis should read it in chunks (`pd.read_csv(..., chunksize=...)`) with only the columns needed (`usecols=`).

**Filename note:** macOS stores Korean filenames in Unicode NFD form, while text typed in code is usually NFC. The two look identical but don't match, so a plain `glob` finds nothing. The cell below normalizes both sides before comparing.


In [32]:
import os
import unicodedata

def find_files(prefix, suffix=".csv", folder="."):
    """Find files whose names start with `prefix`, ignoring NFC/NFD Unicode differences."""
    norm = lambda s: unicodedata.normalize("NFC", s)
    return sorted(
        os.path.join(folder, f) for f in os.listdir(folder)
        if norm(f).startswith(norm(prefix)) and f.endswith(suffix)
    )

trip_files = find_files("서울특별시 공공자전거 대여이력 정보_")
print(f"Found {len(trip_files)} trip file(s):", [unicodedata.normalize("NFC", os.path.basename(f)) for f in trip_files])


Found 1 trip file(s): ['서울특별시 공공자전거 대여이력 정보_전체_2601-2606.csv']


### English Station Lookup

Maps each station ID to its station number, Korean name and romanized English name. If `station_lookup_en.csv` is missing, it is rebuilt from the trip file.

**Note:** the existing lookup has 2,761 stations because it was built from a single month of trips. `station_locations.csv` has 3,430. Future joins between trips and coordinates should use **station IDs** (`rental_station_id` ↔ `대여소_ID`), not station names.


In [33]:
## English station lookup: station ID, station number, Korean name, romanized English name
import os

if os.path.exists("station_lookup_en.csv"):
    stations_en = pd.read_csv("station_lookup_en.csv")
else:
    _stations_raw = pd.read_csv(
        trip_files[-1], encoding="cp949",
        usecols=["대여대여소ID", "대여 대여소번호", "대여 대여소명"],
    ).drop_duplicates(subset=["대여대여소ID"])
    stations_en = _stations_raw.rename(columns={
        "대여대여소ID": "station_id",
        "대여 대여소번호": "station_number",
        "대여 대여소명": "station_name_kr",
    })
    stations_en["station_name_en"] = stations_en["station_name_kr"].apply(romanize)
    stations_en = stations_en.sort_values("station_number")
    stations_en.to_csv("station_lookup_en.csv", index=False)

kr_to_en_station = dict(zip(stations_en["station_name_kr"], stations_en["station_name_en"]))
print(f"{len(stations_en)} stations")
stations_en.head(10)

2761 stations


,station_number,station_name_kr,station_id,station_name_en
0,102,망원역 1번출구 앞,ST-4,Mangwonyeok 1Beonchulgu Ap
1,103,망원역 2번출구 앞,ST-5,Mangwonyeok 2Beonchulgu Ap
2,104,합정역 1번출구 앞,ST-6,Hapjeongyeok 1Beonchulgu Ap
3,105,합정역 5번출구 앞,ST-7,Hapjeongyeok 5Beonchulgu Ap
4,106,합정역 7번출구 앞,ST-8,Hapjeongyeok 7Beonchulgu Ap
5,107,수협 연남동지점,ST-9,Suhyeop Yeonnamdongjijeom
6,108,서교동 사거리,ST-10,Seogyodong Sageori
7,111,상수역 2번출구 앞,ST-15,Sangsuyeok 2Beonchulgu Ap
8,113,홍대입구역 2번출구 앞,ST-18,Hongdaeipguyeok 2Beonchulgu Ap
9,114,홍대입구역 8번출구 앞,ST-20,Hongdaeipguyeok 8Beonchulgu Ap


## Secondary Dataset: Seoul Daily Weather (Jan–Jun 2026)

To study how weather conditions affect bike rental demand, we use daily surface weather observations for Seoul from the Korea Meteorological Administration (KMA) Open Data Portal ([data.kma.go.kr](https://data.kma.go.kr)), Automated Synoptic Observing System (ASOS), station 108 (Seoul).

The data covers the same date range as the bike rental data, **January 1 - June 30, 2026** (181 days), and includes: average/minimum/maximum temperature, daily precipitation, average wind speed, average relative humidity, and total sunshine hours.

Note: `precip_mm` is blank in the raw KMA data on days with no measurable precipitation; we fill these with 0 below.


**Note:** this is a single station (108) observing the whole city once a day, so it gives city-wide conditions, not local ones.


In [34]:
## Load and translate the daily weather dataset
weather = to_english(pd.read_csv("seoul_weather_2026_01-06.csv", encoding="cp949"))
weather["date"] = pd.to_datetime(weather["date"])
weather["precip_mm"] = weather["precip_mm"].fillna(0)
weather.head()


,station_id,station_name_kr,date,avg_temp_c,min_temp_c,max_temp_c,precip_mm,avg_wind_mps,avg_humidity_pct,sunshine_hr
0,108,서울,2026-01-01,-7.4,-10.5,-2.1,0.0,2.8,41.5,8.9
1,108,서울,2026-01-02,-8.0,-11.4,-3.8,0.0,2.7,46.6,9.1
2,108,서울,2026-01-03,-4.7,-9.8,0.0,0.0,1.4,60.0,4.5
3,108,서울,2026-01-04,0.2,-2.7,5.0,0.0,1.9,63.4,7.4
4,108,서울,2026-01-05,-1.9,-4.6,1.8,0.0,2.3,48.5,9.1


In [35]:
## Weather summary statistics
weather.describe()


,station_id,date,avg_temp_c,min_temp_c,max_temp_c,precip_mm,avg_wind_mps,avg_humidity_pct,sunshine_hr
count,181.0,181,181.000000,181.000000,181.000000,181.000000,181.000000,181.000000,181.000000
mean,108.0,2026-04-01 00:00:00,11.030939,6.255249,16.525414,1.623757,2.336464,55.579006,7.573481
min,108.0,2026-01-01 00:00:00,-9.700000,-13.200000,-5.200000,0.000000,1.100000,30.300000,0.000000
25%,108.0,2026-02-15 00:00:00,2.700000,-1.300000,7.500000,0.000000,1.900000,46.600000,4.500000
50%,108.0,2026-04-01 00:00:00,11.900000,7.100000,18.000000,0.000000,2.200000,53.900000,8.900000
75%,108.0,2026-05-16 00:00:00,20.600000,15.100000,26.400000,0.000000,2.600000,63.400000,10.700000
max,108.0,2026-06-30 00:00:00,27.200000,23.000000,34.000000,71.700000,5.300000,90.500000,13.500000
std,0.0,NaN,10.301792,10.017210,10.890986,7.600237,0.584044,12.790309,3.989398


## Spatial Datasets

The four datasets planned for the urban-design phase have now been collected and are loaded below:

1. **Seoul District Boundaries** (`seoul_district_boundaries.geojson`) - 25 district polygons with Korean/English names, sourced from a maintained GitHub mirror (`datainworld/administrative_district`) since Seoul Open Data Plaza had no direct download for this.
2. **Subway Station Coordinates** (`subway_station_locations.csv`) - 784 stations with GPS coordinates, from data.seoul.go.kr (OA-21232).
3. **Seoul Bike Path Network** (`seoul_bike_path_network.geojson`) - 3,535 cycleway segments, pulled from OpenStreetMap via Overpass Turbo since Seoul only publishes fragmented district-level bike-path files, not a citywide one.
4. **Floating Population Estimates** (`seoul_floating_population_2026_01-06.csv`) - daily population estimates by district, Jan-Jun 2026 (4,706 rows), combining domestic and foreign (long/short-term) residents in one dataset.

Also collected: real Ttareungi station GPS coordinates (`station_locations.csv`, 3,430 stations).


### Loading the Spatial Datasets


In [36]:
## Ttareungi station GPS coordinates
stations_geo = pd.read_csv("station_locations.csv", encoding="cp949")
stations_geo = stations_geo.rename(columns={
    "대여소_ID": "station_id",
    "주소1": "address_1",
    "주소2": "address_2",
    "위도": "lat",
    "경도": "lon",
})
print("shape:", stations_geo.shape)
print(stations_geo.dtypes)
print("rows with missing/zero coordinates:", ((stations_geo["lat"] == 0) & (stations_geo["lon"] == 0)).sum())
stations_geo.head()


shape: (3430, 5)
station_id        str
address_1         str
address_2         str
lat           float64
lon           float64
dtype: object
rows with missing/zero coordinates: 77


,station_id,address_1,address_2,lat,lon
0,ST-999,서울특별시 양천구 목동서로 280,목동아파트 8단지 상가동,0.000000,0.000000
1,ST-998,서울특별시 양천구 목동서로 130,목동아파트 4단지 상가동,0.000000,0.000000
2,ST-997,서울특별시 양천구 목동중앙로 49,목동3단지 시내버스정류장,37.534390,126.869598
3,ST-996,서울특별시 양천구 남부순환로88길5-16,양강중학교앞 교차로,37.524334,126.850548
4,ST-995,서울특별시 양천구 중앙로 153 공중화장실,NaN,37.510597,126.857323


**Note:** 77 stations have (0, 0) coordinates and must be excluded from any spatial feature.


In [37]:
## Subway station coordinates
subway = pd.read_csv("subway_station_locations.csv", encoding="cp949")
subway = subway.rename(columns={
    "역사_ID": "station_id",
    "역사명": "station_name",
    "호선": "line",
    "위도": "lat",
    "경도": "lon",
})
print("shape:", subway.shape)
print(subway.dtypes)
print("missing values:\n", subway.isnull().sum())
subway.head()


shape: (784, 5)
station_id        int64
station_name        str
line                str
lat             float64
lon             float64
dtype: object
missing values:
 station_id      0
station_name    0
line            0
lat             0
lon             0
dtype: int64


,station_id,station_name,line,lat,lon
0,9010,동탄,수도권 광역급행철도,37.20034,127.09569
1,9009,구성,수도권 광역급행철도,37.29913,127.10389
2,9008,성남,수도권 광역급행철도,37.39467,127.12058
3,9007,수서,수도권 광역급행철도,37.48637,127.10161
4,9006,삼성,수도권 광역급행철도,37.50887,127.06324


In [38]:
## Seoul district boundaries (GeoJSON)
import json

with open("seoul_district_boundaries.geojson", encoding="utf-8") as f:
    district_geo = json.load(f)

print("feature count:", len(district_geo["features"]))
district_names = [feat["properties"]["SIG_ENG_NM"] for feat in district_geo["features"]]
print("districts:", sorted(district_names))


feature count: 25
districts: ['Dobong-gu', 'Dongdaemun-gu', 'Dongjak-gu', 'Eunpyeong-gu', 'Gangbuk-gu', 'Gangdong-gu', 'Gangnam-gu', 'Gangseo-gu', 'Geumcheon-gu', 'Guro-gu', 'Gwanak-gu', 'Gwangjin-gu', 'Jongno-gu', 'Jung-gu', 'Jungnang-gu', 'Mapo-gu', 'Nowon-gu', 'Seocho-gu', 'Seodaemun-gu', 'Seongbuk-gu', 'Seongdong-gu', 'Songpa-gu', 'Yangcheon-gu', 'Yeongdeungpo-gu', 'Yongsan-gu']


In [39]:
## Seoul bike path network (GeoJSON, from OpenStreetMap)
with open("seoul_bike_path_network.geojson", encoding="utf-8") as f:
    bike_path_geo = json.load(f)

print("feature count:", len(bike_path_geo["features"]))
geom_types = pd.Series([feat["geometry"]["type"] for feat in bike_path_geo["features"] if feat.get("geometry")]).value_counts()
print("geometry types:\n", geom_types)

## Sample of tagged attributes (name, operator, distance in km)
bike_path_props = pd.DataFrame([feat["properties"] for feat in bike_path_geo["features"]])
bike_path_props[["name", "operator", "distance"]].head(10)


feature count: 3536
geometry types:
 LineString    3535
Point            1
Name: count, dtype: int64


,name,operator,distance
0,위례성대로(남측),서울특별시 동부도로사업소,0.208
1,위례성대로(남측),송파구 도시교통과,0.163
2,양재대로,송파구 도시교통과,0.539
3,중랑천서자전거길,NaN,NaN
4,중랑천서자전거길,NaN,NaN
5,중랑천동자전거길,NaN,NaN
6,압구정 나들목,NaN,NaN
7,염창나들목,NaN,NaN
8,중랑천 자전거길 (우이천 합수부),NaN,NaN
9,안양천 자전거길,NaN,NaN


In [40]:
## Floating population estimates (daily, by district, Jan-Jun 2026)
floating_pop = to_english(pd.read_csv("seoul_floating_population_2026_01-06.csv", encoding="utf-8-sig"))
print("shape:", floating_pop.shape)
print(floating_pop.dtypes)
print("missing values:\n", floating_pop.isnull().sum())
floating_pop.head()


shape: (4706, 14)
기준일ID             int64
시구코드              int64
시구명                 str
총생활인구수          float64
내국인생활인구수        float64
장기체류외국인인구수      float64
단기체류외국인인구수      float64
일최대인구수          float64
일최소인구수          float64
주간인구수(09~18)    float64
야간인구수(19~08)    float64
서울외유입인구수        float64
자치구간이동인구수       float64
적재일시              int64
dtype: object
missing values:
 기준일ID           0
시구코드            0
시구명             0
총생활인구수          0
내국인생활인구수        0
장기체류외국인인구수      0
단기체류외국인인구수      0
일최대인구수          0
일최소인구수          0
주간인구수(09~18)    0
야간인구수(19~08)    0
서울외유입인구수        0
자치구간이동인구수       0
적재일시            0
dtype: int64


,기준일ID,시구코드,시구명,총생활인구수,내국인생활인구수,장기체류외국인인구수,단기체류외국인인구수,일최대인구수,일최소인구수,주간인구수(09~18),야간인구수(19~08),서울외유입인구수,자치구간이동인구수,적재일시
0,20260131,11000,서울시,1.143818e+07,1.096621e+07,315876.95616,156099.69409,12450290.55,10654861.65,1.187774e+07,1.112421e+07,1157061.91,2046290.14,20260727
1,20260131,11110,종로구,2.728191e+05,2.524318e+05,12033.35666,8353.85791,410000.27,189673.31,3.436303e+05,2.222396e+05,61411.11,150744.43,20260727
2,20260131,11140,중구,2.592668e+05,2.205160e+05,15726.47166,23024.32625,368309.45,183448.18,3.221502e+05,2.143500e+05,37070.18,139501.36,20260727
3,20260131,11170,용산구,3.148673e+05,2.825674e+05,25773.59041,6526.27583,375403.66,265446.29,3.432500e+05,2.945939e+05,57303.87,87739.95,20260727
4,20260131,11200,성동구,3.430844e+05,3.270154e+05,10896.26500,5172.71750,368714.94,324586.26,3.584763e+05,3.320902e+05,29397.78,79887.00,20260727


**Note:** each day has 26 rows, the 25 districts plus a Seoul-wide total (`시구코드` = 11000). Remove the total row before summing or joining by district. The column names are still in Korean because they aren't in `COLUMN_TRANSLATIONS` yet.


## Station-Level Behaviour Features

One row per rental station (`rental_station_id`), computed over all ~18.5M trips. The trip file is read in chunks with only the needed columns, and the results stay in the `station_features` DataFrame (nothing is written to disk). The daily weather data is joined by date to measure each station's response to cold and rain.

| Feature | Definition |
|---|---|
| `am_peak_share` | weekday rentals 07:00-09:59 ÷ station's weekday rentals |
| `pm_peak_share` | weekday rentals 17:00-19:59 ÷ station's weekday rentals |
| `night_share` | rentals 22:00-04:59 (any day) ÷ station's total rentals |
| `weekend_share` | Saturday/Sunday rentals ÷ station's total rentals |
| `median_duration_min` | exact median of `duration_min` |
| `median_distance_m` | median of `distance_m` (to the nearest 10 m) |
| `round_trip_share` | trips returned to the same station ÷ total rentals |
| `am_net_flow` | (returns − rentals) ÷ (returns + rentals) during weekday 07:00-09:59; negative = bikes leave in the morning |
| `pm_net_flow` | same as above for weekday 17:00-19:59 |
| `log_total_rentals` | log10 of total rentals (station size) |
| `cold_ratio` | rentals on days with average temperature < 0 °C ÷ the station's usual rentals on comparable (weekday/weekend) days; 1 = unaffected, < 1 = use drops |
| `rain_ratio` | same, for days with ≥ 1 mm precipitation (the KMA rain-day threshold) |
| `non_member_share` | rentals by non-members (`user_type` = Non-member: tourists and one-off users) ÷ total rentals |

**Notes:**
- AM and PM net flow are used instead of a single overall net flow, because morning outflows and evening inflows cancel out over a whole day (the overall median is about 0).
- Weather ratios only count days after a station's first rental, so stations that opened mid-period are not penalised for days they did not exist.
- `cold_ratio` and `rain_ratio` are computed here for reference but are **not** used in the PCA or clustering. Linking daily weather to six-month station summaries mixes weather with season and holidays (see the PCA section), so weather is analysed separately after clustering.
- About 30 stations have fewer than 50 rentals, so their shares are unstable. They are removed before PCA.

In [41]:
## Compute station-level features from the trip file (chunked, ~2 min)
import numpy as np

CHUNKSIZE = 1_000_000
FEATURE_COLS = ["rental_datetime", "return_datetime", "duration_min", "distance_m",
                "user_type", "rental_station_id", "return_station_id"]
KR_COLS = [k for k, v in COLUMN_TRANSLATIONS.items() if v in FEATURE_COLS]

date_to_dow = {}  # only ~181 unique dates, so cache their weekday (Mon=0 ... Sun=6)

def hour_and_dow(col):
    """Hour (int) and weekday (Mon=0) from a 'YYYY-MM-DD HH:MM:SS' string column."""
    s = col.str.strip()
    d = s.str[:10]
    for x in d.unique():
        if x not in date_to_dow:
            date_to_dow[x] = pd.to_datetime(x, errors="coerce").dayofweek
    return pd.to_numeric(s.str[11:13], errors="coerce"), d.map(date_to_dow), d

def add(running, new):
    """Add a chunk's counts to the running total."""
    return new if running is None else running.add(new, fill_value=0)

rental_counts = return_counts = duration_freq = distance_freq = daily_counts = None

for chunk in pd.read_csv(trip_files[0], encoding="cp949", usecols=KR_COLS, dtype=str, chunksize=CHUNKSIZE):
    chunk = to_english(chunk).dropna(subset=["rental_station_id"])

    hour, dow, rent_date = hour_and_dow(chunk["rental_datetime"])
    is_weekday = dow <= 4

    ## Rental-side counts per station
    flags = pd.DataFrame({
        "rental_station_id": chunk["rental_station_id"],
        "total_rentals": 1,
        "weekday_rentals": is_weekday,
        "am_peak": is_weekday & hour.between(7, 9),    # 07:00-09:59
        "pm_peak": is_weekday & hour.between(17, 19),  # 17:00-19:59
        "night": (hour >= 22) | (hour <= 4),           # 22:00-04:59
        "weekend": dow >= 5,
        "round_trip": chunk["rental_station_id"] == chunk["return_station_id"],
        "non_member": chunk["user_type"] == "Non-member",
    })
    rental_counts = add(rental_counts, flags.groupby("rental_station_id").sum())

    ## Return-side counts per station (for AM/PM net flow), timed by return_datetime
    ret = chunk.dropna(subset=["return_station_id", "return_datetime"])
    r_hour, r_dow, _ = hour_and_dow(ret["return_datetime"])
    r_weekday = r_dow <= 4
    ret_flags = pd.DataFrame({
        "return_station_id": ret["return_station_id"],
        "total_returns": 1,
        "am_returns": r_weekday & r_hour.between(7, 9),
        "pm_returns": r_weekday & r_hour.between(17, 19),
    })
    return_counts = add(return_counts, ret_flags.groupby("return_station_id").sum())

    ## Frequency tables so exact medians can be found without keeping every row
    station = chunk["rental_station_id"]
    dur = pd.to_numeric(chunk["duration_min"], errors="coerce")
    duration_freq = add(duration_freq, pd.DataFrame({"s": station, "v": dur}).dropna().groupby(["s", "v"]).size())
    dist = (pd.to_numeric(chunk["distance_m"], errors="coerce") / 10).round() * 10   # nearest 10 m
    distance_freq = add(distance_freq, pd.DataFrame({"s": station, "v": dist}).dropna().groupby(["s", "v"]).size())

    ## Rentals per station per day (for the weather ratios)
    daily_counts = add(daily_counts, pd.DataFrame({"s": station, "date": rent_date}).groupby(["s", "date"]).size())

def median_from_freq(freq):
    """Exact median per station from a (station, value) -> count Series."""
    df = freq.rename("n").reset_index().sort_values(["s", "v"])
    def med(g):
        vals, cum = g["v"].to_numpy(), np.cumsum(g["n"].to_numpy())
        total = cum[-1]
        return (vals[np.searchsorted(cum, (total - 1) // 2 + 1)] + vals[np.searchsorted(cum, total // 2 + 1)]) / 2
    return df.groupby("s")[["v", "n"]].apply(med)

## Weather ratios: rentals on cold / rainy days vs. what the station normally does on comparable days
daily = daily_counts.unstack(fill_value=0)                     # stations x dates
daily.columns = pd.to_datetime(daily.columns)
all_dates = pd.date_range(weather["date"].min(), weather["date"].max())
daily = daily.reindex(columns=all_dates, fill_value=0)
first_active = daily.gt(0).idxmax(axis=1)                      # ignore days before a station opened
daily = daily.where(daily.columns.values[None, :] >= first_active.values[:, None])

w = weather.set_index("date").reindex(all_dates)
is_cold = (w["avg_temp_c"] < 0).values                         # 32 days below 0 °C
is_rain = (w["precip_mm"] >= 1).values                         # 29 days with >= 1 mm (KMA rain-day threshold)
is_wkend = (all_dates.dayofweek >= 5)

def weather_ratio(event, baseline):
    """sum(actual on event days) / sum(expected on event days), where 'expected' is the station's
    average on baseline days of the same type (weekday or weekend). 1 = unaffected, < 1 = drops."""
    expected = pd.DataFrame(index=daily.index, columns=daily.columns, dtype=float)
    for wk in (False, True):
        cols = daily.columns[is_wkend == wk]
        base = daily.loc[:, (is_wkend == wk) & baseline].mean(axis=1)
        expected[cols] = np.repeat(base.values[:, None], len(cols), axis=1)
    expected = expected.where(daily.notna())
    actual_sum = daily.loc[:, event].sum(axis=1)
    expected_sum = expected.loc[:, event].sum(axis=1)
    return actual_sum / expected_sum.replace(0, np.nan)

cold_ratio = weather_ratio(is_cold, ~is_cold)
rain_ratio = weather_ratio(is_rain, ~is_rain)

## Build the feature table
feat = rental_counts.join(return_counts, how="outer").fillna(0)
tot = feat["total_rentals"].replace(0, np.nan)
wk = feat["weekday_rentals"].replace(0, np.nan)

def net_flow(returns, rentals):
    return (returns - rentals) / (returns + rentals).replace(0, np.nan)

station_features = pd.DataFrame({
    "total_rentals": feat["total_rentals"].astype(int),
    "total_returns": feat["total_returns"].astype(int),
    "am_peak_share": feat["am_peak"] / wk,
    "pm_peak_share": feat["pm_peak"] / wk,
    "night_share": feat["night"] / tot,
    "weekend_share": feat["weekend"] / tot,
    "median_duration_min": median_from_freq(duration_freq),
    "median_distance_m": median_from_freq(distance_freq),
    "round_trip_share": feat["round_trip"] / tot,
    "am_net_flow": net_flow(feat["am_returns"], feat["am_peak"]),
    "pm_net_flow": net_flow(feat["pm_returns"], feat["pm_peak"]),
    "log_total_rentals": np.log10(feat["total_rentals"].replace(0, np.nan)),
    "cold_ratio": cold_ratio,
    "rain_ratio": rain_ratio,
    "non_member_share": feat["non_member"] / tot,
})
station_features.index.name = "station_id"

## Attach English station names
station_features = (stations_en.set_index("station_id")[["station_number", "station_name_en"]]
                    .join(station_features, how="right")
                    .sort_values("total_rentals", ascending=False))

print(f"{len(station_features):,} stations, {station_features['total_rentals'].sum():,} trips")
station_features.head(10)

2,807 stations, 18,511,943 trips


,station_number,station_name_en,total_rentals,total_returns,am_peak_share,pm_peak_share,night_share,weekend_share,median_duration_min,median_distance_m,round_trip_share,am_net_flow,pm_net_flow,log_total_rentals,cold_ratio,rain_ratio,non_member_share
station_id,,,,,,,,,,,,,,,,,
ST-2031,2715.0,Magoknaruyeok 2Beon Chulgu,72001,73686,0.264174,0.303872,0.080693,0.205817,7.0,1030.0,0.049222,-0.039356,-0.034624,4.857339,0.569187,0.739402,0.003861
ST-3388,5515.0,Hangangbeoseu Mangwon Seonchakjang,64749,61910,0.042674,0.282816,0.127044,0.447358,33.0,4700.0,0.262660,0.050078,0.018762,4.811233,0.085276,0.628531,0.068186
ST-891,1210.0,Rotdewoldeutawo(Jamsilyeok2Beonchulgu Jjok),54503,54275,0.154545,0.401369,0.061666,0.260371,11.0,1350.0,0.061006,0.393792,-0.288165,4.736420,0.416121,0.702263,0.014843
ST-2045,2728.0,Magoknaruyeok 3Beon Chulgu,52801,53364,0.242560,0.313324,0.139846,0.172743,7.0,1110.0,0.040094,-0.076618,0.122298,4.722642,0.543582,0.662220,0.004981
ST-1718,2701.0,Magoknaruyeok 5Beonchulgu Dwipyeon,49969,51973,0.419229,0.267952,0.042426,0.148052,6.0,870.0,0.029338,-0.033141,-0.058753,4.698701,0.595286,0.736197,0.002842
ST-3402,5891.0,Hangangbeoseu Yeouido Seonchakjang,49172,44153,0.022175,0.320744,0.134975,0.431384,42.0,4560.0,0.390120,-0.041142,-0.089275,4.691718,0.088397,0.581808,0.109676
ST-1065,1153.0,"Balsanyeok 1Beon, 9Beon Ingeun Daeyeoso",42823,43084,0.157872,0.256363,0.192537,0.233940,8.0,1190.0,0.046003,-0.086086,0.169013,4.631677,0.541360,0.706821,0.005371
ST-1720,2622.0,Olrimpikgongwonyeok 3Beonchulgu,37783,37558,0.191710,0.289630,0.083450,0.326972,12.0,1560.0,0.126247,0.123044,-0.070650,4.577296,0.348023,0.696184,0.018368
ST-3390,5153.0,Hangangbeoseu Ttukseom Seonchakjang,37407,37011,0.033850,0.314469,0.144679,0.441655,36.0,4060.0,0.361296,0.180290,-0.030679,4.572953,0.114090,0.497068,0.092924


In [42]:
## Feature summary statistics
station_features.describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
station_number,2761.0,2501.907,1667.470,102.000,1056.000,2198.000,4020.000,9980.000
total_rentals,2807.0,6594.921,6080.391,0.000,2669.000,4921.000,8559.500,72001.000
total_returns,2807.0,6559.222,6339.567,1.000,2352.000,4795.000,8681.000,73686.000
am_peak_share,2795.0,0.201,0.096,0.000,0.131,0.190,0.264,1.000
pm_peak_share,2795.0,0.247,0.086,0.000,0.191,0.232,0.288,1.000
night_share,2800.0,0.115,0.061,0.000,0.077,0.108,0.144,1.000
weekend_share,2800.0,0.255,0.070,0.000,0.225,0.257,0.286,1.000
median_duration_min,2800.0,12.030,5.100,1.000,9.000,11.000,14.000,62.000
median_distance_m,2800.0,1546.807,659.750,0.000,1180.000,1410.000,1700.000,11695.000
round_trip_share,2800.0,0.094,0.044,0.000,0.068,0.087,0.113,0.930


# Principal Component Analysis (PCA)

**Goal:** the station features overlap (for example, stations with a high weekend share also tend to have long, round trips). PCA combines correlated features into a few uncorrelated components that summarise how stations differ. These components are then used as input for clustering.

This section follows the steps of CDS 303 Demo 6.

In [43]:
## Setup
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings(action = 'ignore')

## 1. Input Variables and Preprocessing

### Variables

We start with 11 candidate station-level variables that describe **how** a station is used. Spatial context (subway distance, bike paths, district) is deliberately left out here. It is kept for the later classification step, which explains the clusters, so the explanation is not circular.

| Group | Variables |
|---|---|
| Time of day | `am_peak_share`, `pm_peak_share`, `night_share`, `weekend_share` |
| Trip type | `median_duration_min`, `median_distance_m`, `round_trip_share` |
| Bike flow | `am_net_flow`, `pm_net_flow` |
| Station size | `log_total_rentals` |
| Rider mix | `non_member_share` |

### Why the weather variables are not included

An earlier version of this PCA also included `cold_ratio` and `rain_ratio`. They were removed because linking daily weather to station-level summaries is temporally tricky:

- **Season and weather are mixed together.** All 32 days below 0 °C fall in January-February, while ridership rises steadily into spring (warmer days, longer daylight, new stations). A station's lower use on cold days is therefore partly a winter-vs-spring effect, not only a temperature effect.
- **Holidays fall in the same period.** Lunar New Year lowers commuter trips in mid-February for reasons unrelated to weather.
- **Daily weather is coarse.** Daily precipitation does not show *when* it rained (overnight rain barely affects riding, rain at rush hour does), and there is only one citywide weather station.
- **They added little shared structure.** In the earlier run, `rain_ratio` formed a component almost on its own (loading 0.90, eigenvalue 0.99), and `cold_ratio` mostly duplicated the leisure pattern already captured by weekend share and trip length.

Weather sensitivity is still part of our research question. It will be analysed **after** clustering, by modelling each cluster's daily rentals against temperature and rain while controlling for month, day of the week and public holidays. This keeps the clusters based only on usage behaviour and gives a cleaner estimate of the weather effect.

### Preprocessing steps

1. **Low-volume stations:** stations with fewer than 50 rentals in six months are removed (30 stations). Their shares are based on too few trips (many are exactly 0 or 1) and would act as outliers. This leaves **2,777 stations**.
2. **Missing values:** after that filter, none of the 11 variables has missing values, so no imputation is needed. (Before the filter, a few stations with no weekday rentals had undefined peak shares; the filter removes them.)
3. **Redundant variables:** the correlation matrix below shows `median_duration_min` and `median_distance_m` are almost the same measurement (r = 0.91). Keeping both would give "trip length" double weight in PCA, so `median_distance_m` is dropped. No other pair exceeds |r| = 0.74. This leaves **10 variables**.
4. **Outliers:** several variables are strongly right-skewed because of a few extreme stations (e.g. `round_trip_share` and `non_member_share` both have a skew of about 4.4). Each variable is clipped at its 1st and 99th percentiles (winsorizing) so a handful of stations cannot dominate a component.
5. **Measurement scale:** the variables are in different units (minutes, proportions from 0-1, net flow from −1 to 1, log counts). PCA is driven by variance, so unscaled variables with larger ranges would dominate. All variables are standardized with `StandardScaler` (mean 0, standard deviation 1).

In [ ]:
## Candidate variables and low-volume / missing-value filtering
CANDIDATE_VARS = ["am_peak_share", "pm_peak_share", "night_share", "weekend_share",
                  "median_duration_min", "median_distance_m", "round_trip_share",
                  "am_net_flow", "pm_net_flow", "log_total_rentals",
                  "non_member_share"]   # cold_ratio / rain_ratio excluded (see note above)

MIN_RENTALS = 50
active = station_features[station_features["total_rentals"] >= MIN_RENTALS]
print(f"Stations: {len(station_features):,} total, {len(active):,} with >= {MIN_RENTALS} rentals")
print("Missing values after the volume filter:")
print(active[CANDIDATE_VARS].isnull().sum()[lambda s: s > 0])

pca_input = active.dropna(subset=CANDIDATE_VARS)
print(f"Stations used for PCA: {len(pca_input):,}")

In [ ]:
## Correlation between candidate variables
cor = pca_input[CANDIDATE_VARS].corr()

sns.clustermap(cor, annot = True, fmt = ".2f", cmap = 'RdYlBu_r', vmin = -1, vmax = 1, figsize = (11, 11), annot_kws = {"size": 7})
plt.show()

In [ ]:
## Drop the redundant variable, clip outliers, and standardize
PCA_VARS = [v for v in CANDIDATE_VARS if v != "median_distance_m"]   # r = 0.91 with median_duration_min

station_raw = pca_input[PCA_VARS]
station_clipped = station_raw.clip(station_raw.quantile(0.01), station_raw.quantile(0.99), axis = 1)

print("Skewness before and after clipping:")
print(pd.DataFrame({"before": station_raw.skew(), "after": station_clipped.skew()}).round(2))

station_st = StandardScaler().fit_transform(station_clipped)
station_st = pd.DataFrame(station_st, columns = PCA_VARS, index = station_clipped.index)
station_st.head(10)

## 2. Component Selection

In [ ]:
pca = PCA(random_state = 2000, n_components = None)
pca.fit(station_st)

## Variance (eigenvalues) and explained variance ratio of each component
pc_list = ["PC" + str(i) for i in range(1, len(PCA_VARS) + 1)]
variance_df = pd.DataFrame({
    "eigenvalue": pca.explained_variance_,
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative_ratio": np.cumsum(pca.explained_variance_ratio_),
}, index = pc_list)
variance_df.round(3)

In [ ]:
## Scree plot: eigenvalue of each component, with the Kaiser criterion (eigenvalue = 1)
x_axis = range(1, len(PCA_VARS) + 1)
plt.figure(figsize = (8, 5))
plt.plot(x_axis, pca.explained_variance_, marker = 'o')
plt.axhline(1, color = 'red', linestyle = '--', label = 'Kaiser criterion (eigenvalue = 1)')
plt.xticks(x_axis)
plt.xlabel('Principal Components')
plt.ylabel('Eigenvalue (variance)')
plt.title('Scree Plot')
plt.legend()
plt.grid()
plt.show()

In [ ]:
## Explained variance ratio of each component (bars) and cumulative explained variance (red line), Demo 6 plot
exp_variance_ratio = pca.explained_variance_ratio_

def explained_variance_ratio_plot(exp_variance_ratio):
    x_axis = range(1, len(exp_variance_ratio)+1)
    plt.figure(figsize = (8, 5))
    plt.bar(x_axis, exp_variance_ratio,
            align = 'center', label = 'Individual Explained Variance Ratio')
    plt.step(x_axis, np.cumsum(exp_variance_ratio),
             where = 'mid', color='red', label='Cumulative Explained Variance Ratio')
    plt.axhline(0.8, color = 'grey', linestyle = ':', label = '80% of variance')
    plt.axvline(4.5, color = 'green', linestyle = '--', label = 'Components retained (PC1-PC4)')
    plt.ylim(0, 1.1)
    plt.xticks(x_axis)
    plt.xlabel('Principal Components')
    plt.ylabel('Explained Variance Ratio')
    plt.title('Individual and Cumulative Explained Variance')
    plt.legend(loc = 'center right')
    plt.grid()
    plt.show()

explained_variance_ratio_plot(exp_variance_ratio)

### Explained variance

| Component | Eigenvalue | Explained variance | Cumulative |
|---|---|---|---|
| PC1 | 3.18 | 31.8% | 31.8% |
| PC2 | 2.53 | 25.3% | 57.0% |
| PC3 | 1.27 | 12.7% | 69.8% |
| PC4 | 1.13 | 11.3% | **81.0%** |
| PC5 | 0.62 | 6.2% | 87.2% |
| PC6-PC10 | < 0.38 each | 12.8% combined | 100% |

### How many components to keep: **4**

Three common rules all point to the same answer:

- **Kaiser criterion:** exactly four components (PC1-PC4) have eigenvalues above 1, meaning each explains more variance than a single original variable. PC5 is well below (0.62).
- **Cumulative variance:** four components explain **81.0%** of the variance, above the commonly used 80% threshold, while reducing 10 variables to 4.
- **Scree plot:** there are two clear drops: after PC2 (2.53 → 1.27) and after PC4 (1.13 → 0.62). After PC4 the curve is flat, and each further component adds 6% or less.

PC1 and PC2 are the main dimensions (57% together); PC3 and PC4 add smaller but clearly interpretable patterns. Adding PC5 would only add 6.2% and would break the Kaiser rule, so four components is the best balance between simplicity and information kept.

## 3. Component Interpretation

The sign of a principal component is arbitrary (PCA can return a component or its mirror image). To make the interpretation stable, each component is oriented so that a chosen reference variable has a positive loading: `pm_peak_share` for PC1, `weekend_share` for PC2, `night_share` for PC3, `log_total_rentals` for PC4 and `round_trip_share` for PC5. Flipping a sign does not change the variance explained or the analysis.

In [ ]:
## Orient the signs of the first five components so they read the same way every run
ANCHORS = ["pm_peak_share", "weekend_share", "night_share", "log_total_rentals", "round_trip_share"]
for i, var in enumerate(ANCHORS):
    if pca.components_[i, PCA_VARS.index(var)] < 0:
        pca.components_[i] *= -1

N_KEEP = 4

## Loadings (eigenvectors): one column per component
loadings_df = pd.DataFrame(pca.components_.T, columns = pc_list, index = PCA_VARS)
loadings_df.index.name = "variable"
loadings_df.iloc[:, :N_KEEP + 1].round(2)

In [ ]:
## Loading heatmap for the retained components (plus PC5 for reference)
plt.figure(figsize = (7, 7))
ax = sns.heatmap(loadings_df.iloc[:, :N_KEEP + 1], annot = True, fmt = ".2f", cmap = 'Spectral', center = 0, vmin = -1, vmax = 1)
plt.title('PCA Loadings')
plt.tight_layout()
plt.show()

In [ ]:
## Strongest contributors to each retained component (|loading| >= 0.3)
for pc in pc_list[:N_KEEP]:
    top = loadings_df[pc][loadings_df[pc].abs() >= 0.3].sort_values(key = abs, ascending = False)
    print(f"{pc} ({pca.explained_variance_ratio_[pc_list.index(pc)]*100:.1f}%):",
          ", ".join(f"{v} ({w:+.2f})" for v, w in top.items()))

In [ ]:
## Projected data (scores) for each station
scores_df = pd.DataFrame(pca.transform(station_st), columns = pc_list, index = station_st.index)
scores_df.insert(0, "station_name_en", pca_input["station_name_en"])
scores_df.insert(1, "total_rentals", pca_input["total_rentals"])

## Stations with the highest and lowest scores on each retained component (busy stations only, for readability)
busy = scores_df[scores_df["total_rentals"] >= 2000]
for pc in pc_list[:N_KEEP]:
    print(f"\n{pc} highest:", ", ".join(busy.nlargest(4, pc)["station_name_en"].fillna("(no name)")))
    print(f"{pc} lowest: ", ", ".join(busy.nsmallest(4, pc)["station_name_en"].fillna("(no name)")))

In [ ]:
## Confirm that the principal components are uncorrelated with each other
cor = scores_df[pc_list].corr()
sns.clustermap(cor, annot = False, cmap = 'RdYlBu_r', vmin = -1, vmax = 1, figsize = (4, 4))
plt.show()

### What the components represent

**PC1: Home end vs. work end of the commute (31.8%)**
- Positive: `pm_peak_share` (+0.50), `am_net_flow` (+0.49).
- Negative: `am_peak_share` (−0.47), `pm_net_flow` (−0.46).
- A high score means bikes **arrive in the morning and leave in the evening**: a workplace station. The highest scores are offices and financial institutions, mostly in Yeouido and the Gasan Digital Complex: Korea Development Bank, World Meridian Venture Center, the Financial Supervisory Service, Byucksan Digital Valley and Woori Bank.
- A low score means the reverse: bikes **leave in the morning and come back in the evening**, the pattern of residential areas. The lowest-scoring stations are almost all apartment complexes and housing (Godeok Raemian Hillstate, Gongneung Pungrim, Mokdong apartments).

**PC2: Leisure vs. everyday utility (25.3%)**
- Positive: `median_duration_min` (+0.51), `weekend_share` (+0.50), `non_member_share` (+0.48), `round_trip_share` (+0.39).
- A high score means long weekend round trips, often by casual (non-member) riders. The highest scores are Han River stations: the Mangwon, Ttukseom and Yeouido ferry docks, plus riverside stations at Sinbanpo and Yeouido.
- A low score means short weekday trips by regular members. The lowest scores are all corporate campuses in Magok (LG Science Park buildings, LG U+, Lotte R&D Center, Kolon One&Only Tower).

**PC3: Night-time use (12.7%)**
- Positive: `night_share` (+0.75); negative: `round_trip_share` (−0.47), `log_total_rentals` (−0.28), `am_peak_share` (−0.26).
- High scores are stations used late at night, such as Itaewon Station exit 3, Samgakji Station exit 3 and Hansung University Station exit 7 (nightlife and student areas). Low scores are daytime park and neighbourhood stations such as World Cup Park and Mokdong apartments.

**PC4: Station size (11.3%)**
- Dominated by `log_total_rentals` (+0.83), with `night_share` (+0.32).
- This separates very busy stations (e.g. Bongrimgyo traffic island, Sillim Station exit 8, Omokgyo Station exit 3) from quiet ones, independently of *how* they are used. A busy commuter station and a quiet one of the same type need very different numbers of bikes.

**Note:** these descriptions explain what *high and low scores* on each component mean, using the most extreme stations as examples. Each component is a continuous direction along which stations vary; it does not divide stations into groups. Whether distinct station types exist is tested in the clustering step.

## 4. Visualization and Discussion

### Biplot of PC1 and PC2

Each point is a station (scores scaled to fit the plot, as in Demo 6). Each red arrow is a variable: its direction shows which way that variable increases across the plot, and its length shows how strongly it contributes to these two components. A few well-known stations are labelled for orientation.

**How to read this plot:**
- PCA identifies **directions of variation**, not clusters. Any visual groupings in the scatter are not confirmed station types; clustering will test that separately.
- The plot shows only **PC1 and PC2 (57.0% of the variance)**. Four components are retained for later analysis (81.0%), so PC3 (night-time use) and PC4 (station size) are not visible here. Two stations that look close together in this plot can still differ on PC3 and PC4.

In [ ]:
## biplot: https://sukhbinder.wordpress.com/2015/08/05/biplot-with-python/ (from Demo 6, with labelled example stations)
def biplot(score, coeff, pcax, pcay, labels=None, highlight=None):
  pca1 = pcax - 1
  pca2 = pcay - 1

  xs = score[:,pca1]
  ys = score[:,pca2]

  scalex = 1.0/(xs.max()- xs.min())
  scaley = 1.0/(ys.max()- ys.min())

  plt.figure(figsize=(11, 9))
  plt.scatter(xs*scalex, ys*scaley, alpha = 0.3, s = 10)

  n = coeff.shape[0]
  for i in range(n):
    plt.arrow(0, 0, coeff[i,pca1], coeff[i,pca2],color='r',alpha=0.5)
    if labels is None:
      plt.text(coeff[i,pca1]* 1.15, coeff[i,pca2] * 1.15, "Var"+str(i+1), color='g', ha='center', va='center')
    else:
      plt.text(coeff[i,pca1]* 1.15, coeff[i,pca2] * 1.15, labels[i], color='g', ha='center', va='center')

  ## label a few example stations: highlight = {row position: label}
  if highlight is not None:
    for pos, name in highlight.items():
      plt.scatter(xs[pos]*scalex, ys[pos]*scaley, color = 'black', s = 30)
      plt.annotate(name, (xs[pos]*scalex, ys[pos]*scaley), xytext = (5, 5), textcoords = 'offset points', fontsize = 9)

  plt.xlim(-0.7, 0.7)
  plt.ylim(-0.6, 0.8)

  plt.xlabel(f"PC{pcax} ({pca.explained_variance_ratio_[pca1]*100:.2f} %): home end  <->  work end")
  plt.ylabel(f"PC{pcay} ({pca.explained_variance_ratio_[pca2]*100:.2f} %): everyday utility  <->  leisure")
  plt.grid()

EXAMPLES = {
    "ST-3388": "Mangwon Han River dock",
    "ST-3251": "LG Science Park E14",
    "ST-2419": "Sumyeongsan Park apartments",
    "ST-2694": "National Assembly gate 6",
    "ST-2031": "Magoknaru Stn exit 2",
}
highlight = {station_st.index.get_loc(s): name for s, name in EXAMPLES.items() if s in station_st.index}

scores = pca.transform(station_st)
biplot(scores, pca.components_.T, 1, 2, labels = PCA_VARS, highlight = highlight)
plt.show()

### Visible patterns

- **Opposing arrows show the commute direction (PC1).** `am_peak_share` and `pm_net_flow` point left, while `pm_peak_share` and `am_net_flow` point right. These are two sides of the same daily cycle: stations further left lose bikes in the morning and regain them in the evening; stations further right do the reverse. Sumyeongsan Park apartments lie toward the left; LG Science Park lies far to the right.
- **Leisure variables point upward (PC2).** `weekend_share`, `median_duration_min`, `non_member_share` and `round_trip_share` all point up, so stations higher in the plot have longer, weekend, round trips by casual riders. The Mangwon Han River dock is at the very top.
- **The two directions combine.** Because PC1 and PC2 are independent, a station can score high on both. The National Assembly gate, for example, is high on PC1 (a workplace on weekdays) and also high on PC2 (it is next to Yeouido Han River Park, so it attracts weekend trips). Magoknaru Station exit 2, a busy transfer point, is near zero on PC1 because bikes flow both in and out during each peak.
- **Uneven spread.** Most stations are concentrated near the centre and left, with ordinary, moderately commuter-oriented usage. Fewer stations extend far to the right or toward the top, and the spread toward high PC2 is made up of a small number of extreme stations (122 score above 3 on PC2, mostly by the river). This shows how usage *varies*: it does not by itself show separate groups, and the points form a continuous spread without clear gaps.
- **Short arrows** (`log_total_rentals`, `night_share`) contribute little to PC1 and PC2. Their variation is captured by PC3 and PC4, which this plot does not show.

### Relation to the research question and intended audience

Our research question asks what distinct usage profiles Ttareungi stations show and which characteristics best explain them. PCA does not answer the first part on its own, since it finds directions of variation rather than profiles. It does show **which characteristics drive the differences between stations**: mainly *where a station sits in the daily commute* (home end vs. work end, PC1) and *whether it serves leisure or everyday trips* (PC2), which together explain 57% of the variation. Night-time use (PC3) and station size (PC4) are secondary directions, and the four together explain 81%. These four components are the dimensions on which the clustering step will look for distinct profiles.

For the intended audience, **Seoul Facilities Corporation, which operates Ttareungi**, each direction corresponds to an operational question:
- **PC1 (home ↔ work) relates to rebalancing.** Stations at opposite ends empty and fill in opposite directions each day, so bikes need to be moved from high-PC1 stations back toward low-PC1 stations before the morning peak.
- **PC2 (leisure ↔ utility) relates to timing of demand.** High-PC2 stations, such as the Han River docks, need bikes on weekends, while low-PC2 stations need them on weekday mornings and evenings.
- **PC4 (size)** relates to how many bikes and docks a station needs, independently of how it is used.

### Next step

The four retained component scores (PC1-PC4, 81.0% of the variance) will be used as input for clustering, which will test whether stations form distinct profiles along these directions. Because the scores are standardized and uncorrelated, distance-based methods such as k-means treat each dimension fairly. The clusters will then be explained with a classifier and SHAP values, using both the original features and spatial context (distance to subway, bike paths, district). Weather sensitivity will be measured for each cluster with a daily model that controls for month, day of week and holidays.

### Limitations

- The data covers January-June only, so the summer peak and the July monsoon are not included, and summarising six months into one row per station hides any seasonal change in how a station is used.
- `non_member_share` and `round_trip_share` remain right-skewed after clipping (skew about 2.5 and 1.1), so a few leisure stations still have strong influence on PC2.
- PCA finds linear combinations only. Some station patterns may be non-linear and will need to be checked in the clustering step.